<a href="https://colab.research.google.com/github/bruan789/reddit-video-automation/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
import hashlib
import os
import secrets
import shutil
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "fastapi",
        "uvicorn",
        "pyngrok",
        "moviepy<2",
        "pillow",
        "rembg[cpu]",
        "imageio-ffmpeg",
        "python-multipart",
        "requests",
    ]
)

import gc
import importlib
import importlib.util
import json
import re
import tempfile
import threading
import time
import traceback
from functools import lru_cache
from io import BytesIO
from pathlib import Path
from typing import Any

import requests
import uvicorn
from PIL import Image
from fastapi import BackgroundTasks, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse
from pyngrok import ngrok
from starlette.background import BackgroundTask


def ask_secret(prompt: str) -> str:
    value = getpass.getpass(prompt).strip()
    if not value:
        raise ValueError(f"El valor solicitado es obligatorio: {prompt}")
    return value


def find_nvenc_ffmpeg() -> str:
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        raise RuntimeError("No se encontró FFmpeg. Selecciona un entorno de ejecución con GPU NVIDIA.")
    encoders = subprocess.run(
        [ffmpeg, "-hide_banner", "-encoders"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout
    if "h264_nvenc" not in encoders:
        raise RuntimeError(
            "FFmpeg no ofrece h264_nvenc. En Colab selecciona Entorno de ejecución > "
            "Cambiar tipo de entorno de ejecución > GPU NVIDIA y vuelve a ejecutar la celda."
        )
    probe = subprocess.run(
        [
            ffmpeg,
            "-hide_banner",
            "-loglevel",
            "error",
            "-f",
            "lavfi",
            "-i",
            "color=c=black:s=256x256:d=0.1",
            "-frames:v",
            "1",
            "-an",
            "-c:v",
            "h264_nvenc",
            "-f",
            "null",
            "-",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode:
        raise RuntimeError(
            "La GPU NVIDIA o el codificador h264_nvenc no están disponibles en esta sesión: "
            f"{probe.stderr.strip()}"
        )
    return ffmpeg


print("Validando aceleración NVIDIA h264_nvenc...")
FFMPEG_PATH = find_nvenc_ffmpeg()
os.environ["IMAGEIO_FFMPEG_EXE"] = FFMPEG_PATH

NGROK_TOKEN = ask_secret("Token de autenticación de ngrok (entrada oculta): ")
FACEBOOK_PAGE_ID = input("ID de la página de Facebook: ").strip()
FACEBOOK_PAGE_ACCESS_TOKEN = ask_secret("Token de acceso de la página de Facebook (entrada oculta): ")
API_KEY = secrets.token_urlsafe(32)
if not FACEBOOK_PAGE_ID:
    raise ValueError("El ID de la página de Facebook es obligatorio.")

ngrok.set_auth_token(NGROK_TOKEN)
PUBLICATION_URL = "https://graph-video.facebook.com/v23.0"
WORK_ROOT = Path(tempfile.mkdtemp(prefix="colab_video_jobs_"))
app = FastAPI(title="ALEXAM Tech Studio Colab renderer")
jobs: dict[str, dict[str, Any]] = {}
jobs_lock = threading.Lock()
render_lock = threading.Lock()


def authorized(api_key: str | None) -> None:
    if not api_key or not secrets.compare_digest(api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Clave de API no válida.")


def update_job(job_id: str, **values: Any) -> None:
    with jobs_lock:
        jobs[job_id].update(values)


async def save_upload(upload: UploadFile, destination: Path, limit_bytes: int) -> None:
    size = 0
    with destination.open("wb") as output:
        while chunk := await upload.read(1024 * 1024):
            size += len(chunk)
            if size > limit_bytes:
                raise HTTPException(status_code=413, detail=f"Archivo demasiado grande: {upload.filename}")
            output.write(chunk)
    if not size:
        raise HTTPException(status_code=400, detail=f"Archivo vacío: {upload.filename}")


def clean_text(value: str) -> str:
    text = re.sub(r"^\s*T[ÍI]TULO\s*:\s*.*$", "", value, flags=re.IGNORECASE | re.MULTILINE)
    text = re.split(r"^\s*HASHTAGS\s*:\s*", text, maxsplit=1, flags=re.IGNORECASE | re.MULTILINE)[0]
    return re.sub(r"\n{3,}", "\n\n", text).strip()


@lru_cache(maxsize=1)
def get_rembg_session():
    from rembg import new_session

    return new_session("u2net")


def process_overlay_image(image_path: Path, cache_dir: Path) -> Path:
    source_bytes = image_path.read_bytes()
    with Image.open(BytesIO(source_bytes)) as source_image:
        if source_image.convert("RGBA").getchannel("A").getextrema()[0] < 255:
            return image_path

    from rembg import remove

    result_path = cache_dir / f"{hashlib.sha256(source_bytes).hexdigest()}.png"
    if not result_path.is_file():
        cache_dir.mkdir(parents=True, exist_ok=True)
        cutout_bytes = remove(source_bytes, session=get_rembg_session())
        with Image.open(BytesIO(cutout_bytes)) as cutout:
            cutout.convert("RGBA").save(result_path, format="PNG")
    return result_path


def normalize_word(value: str) -> str:
    return re.sub(r"[^\wáéíóúüñ]", "", value.casefold())


def word_intervals(text: str, duration: float, timing_path: Path | None) -> list[tuple[float, float]]:
    words = text.split()
    if not words:
        return []
    if timing_path and timing_path.exists():
        boundaries = json.loads(timing_path.read_text(encoding="utf-8"))
        intervals: list[tuple[float, float]] = []
        boundary_index = 0
        for word in words:
            expected = normalize_word(word)
            while boundary_index < len(boundaries):
                boundary = boundaries[boundary_index]
                boundary_index += 1
                if normalize_word(str(boundary.get("text", ""))) == expected:
                    start = max(0.0, float(boundary["offset"]) / 10_000_000)
                    end = min(duration, start + float(boundary["duration"]) / 10_000_000)
                    intervals.append((start, max(start, end)))
                    break
            else:
                intervals = []
                break
        if len(intervals) == len(words) and any(end > start for start, end in intervals):
            return intervals

    weights = [max(1, len(word)) + (4 if re.search(r"[,;:]$", word) else 0) + (9 if re.search(r"[.!?…]$", word) else 0) for word in words]
    total_weight = sum(weights)
    cursor = 0.0
    result = []
    for weight in weights:
        end = cursor + duration * weight / total_weight
        result.append((cursor, end))
        cursor = end
    return result


def render_video(
    story: str,
    audio_path: Path,
    timing_path: Path | None,
    background_path: Path | None,
    font_path: Path | None,
    overlay_paths: list[Path],
    output_path: Path,
    style: dict[str, Any],
    progress,
) -> None:
    from PIL import Image, ImageDraw, ImageFont

    if not hasattr(Image, "ANTIALIAS"):
        Image.ANTIALIAS = Image.Resampling.LANCZOS

    from moviepy.editor import (
        AudioFileClip,
        ColorClip,
        CompositeVideoClip,
        ImageClip,
        VideoFileClip,
        vfx,
    )
    from proglog import ProgressBarLogger

    class RenderProgressLogger(ProgressBarLogger):
        def bars_callback(self, bar, attr, value, _old_value=None):
            if bar == "t" and attr == "index":
                total = self.bars[bar].get("total") or 1
                progress(20 + 50 * value / total)

    narration = AudioFileClip(str(audio_path))
    source_background = None
    background = None
    final = None
    caption_clips = []
    overlay_clips = []
    try:
        duration = float(narration.duration)
        if duration <= 0:
            raise ValueError("El audio no contiene una duración válida.")
        width, height = 1080, 1920
        if background_path:
            if background_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
                source_background = ImageClip(str(background_path)).set_duration(duration)
            else:
                source_background = VideoFileClip(
                    str(background_path),
                    audio=False,
                    target_resolution=(width, height),
                )
            background = source_background.resize(height=height)
            if background.w < width:
                background = background.resize(width=width)
            background = background.crop(
                x_center=background.w / 2,
                y_center=background.h / 2,
                width=width,
                height=height,
            )
            background = background.fx(vfx.loop, duration=duration).subclip(0, duration)
        else:
            background = ColorClip((width, height), color=(0, 0, 0), duration=duration)

        overlay_styles = style.get("overlays")
        if not isinstance(overlay_styles, list):
            overlay_styles = [style.get("overlay") or {}]
        for index, overlay_path in enumerate(overlay_paths):
            overlay_style = (
                overlay_styles[index]
                if index < len(overlay_styles) and isinstance(overlay_styles[index], dict)
                else {}
            )
            processed_overlay = process_overlay_image(
                overlay_path,
                output_path.parent / "overlay_cache",
            )
            overlay = ImageClip(str(processed_overlay), transparent=True)
            overlay = overlay.resize(
                width=max(1, int(overlay.w * max(1, int(overlay_style.get("scale", 100))) / 100))
            ).set_duration(duration).set_position(
                (
                    int(overlay_style.get("x", 0)),
                    int(overlay_style.get("y", 0)),
                )
            )
            overlay_clips.append(overlay)

        story_words = clean_text(story).split()
        intervals = word_intervals(" ".join(story_words), duration, timing_path)
        font_size = max(28, min(160, int(style.get("font_size", 48))))
        requested_y = max(0, min(height - 320, int(style.get("text_y", 1590))))
        text_y = max(0, min(height - 320, requested_y - 120))
        if font_path and font_path.exists():
            caption_font = ImageFont.truetype(str(font_path), font_size)
        else:
            caption_font = ImageFont.truetype(
                "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                font_size,
            )

        caption_dir = output_path.parent / "captions"
        caption_dir.mkdir()
        max_words = 2 if "viral" in str(style.get("template", "")).casefold() else 5
        chunks = []
        for word_index in range(0, len(story_words), max_words):
            start = intervals[word_index][0]
            last_word_index = min(len(intervals) - 1, word_index + max_words - 1)
            end = min(duration, intervals[last_word_index][1])
            if end > start:
                chunks.append((start, end, story_words[word_index : word_index + max_words]))
        for index, (start, end, words) in enumerate(chunks):
            image = Image.new("RGBA", (width, 320), (0, 0, 0, 0))
            draw = ImageDraw.Draw(image)
            caption = " ".join(words)
            if style.get("emoji"):
                lowered = caption.casefold()
                emoji = (
                    "😱" if any(term in lowered for term in ("traición", "mentira", "secreto"))
                    else "💔" if any(term in lowered for term in ("llor", "dolor", "triste"))
                    else "✨"
                )
                caption = f"{caption} {emoji}"
            box = draw.textbbox((0, 0), caption, font=caption_font, stroke_width=4)
            x = max(12, (width - (box[2] - box[0])) // 2)
            y = max(8, (320 - (box[3] - box[1])) // 2 - box[1])
            color = "#FFE600" if "amarillo" in str(style.get("template", "")).casefold() else "#FFFFFF"
            draw.text(
                (x, y),
                caption,
                font=caption_font,
                fill=color,
                stroke_width=4,
                stroke_fill="#111111",
            )
            image_path = caption_dir / f"caption_{index:05d}.png"
            image.save(image_path)
            image.close()
            caption_clips.append(
                ImageClip(str(image_path))
                .set_start(start)
                .set_duration(end - start)
                .set_position((0, text_y))
            )
            if index % 25 == 0:
                progress(5 + 15 * index / max(1, len(chunks)))

        final = CompositeVideoClip(
            [background, *overlay_clips, *caption_clips],
            size=(width, height),
            use_bgclip=True,
        ).set_audio(narration)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        final.write_videofile(
            str(output_path),
            fps=30,
            codec="h264_nvenc",
            audio_codec="aac",
            audio_bitrate="192k",
            preset="p4",
            threads=2,
            ffmpeg_params=["-cq", "19", "-b:v", "0", "-pix_fmt", "yuv420p", "-movflags", "+faststart"],
            temp_audiofile=str(output_path.parent / "audio_temp.m4a"),
            remove_temp=True,
            logger=RenderProgressLogger(),
        )
        progress(75)
    finally:
        if final is not None:
            final.close()
        for clip in caption_clips:
            clip.close()
        for clip in overlay_clips:
            clip.close()
        if background is not None:
            background.close()
        if source_background is not None:
            source_background.close()
        narration.close()
        gc.collect()
        if importlib.util.find_spec("torch"):
            torch = importlib.import_module("torch")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def publish_video(video_path: Path, description: str) -> str:
    with video_path.open("rb") as video_file:
        response = requests.post(
            f"{PUBLICATION_URL}/{FACEBOOK_PAGE_ID}/videos",
            data={
                "access_token": FACEBOOK_PAGE_ACCESS_TOKEN,
                "description": description,
                "published": "true",
            },
            files={"source": (video_path.name, video_file, "video/mp4")},
            timeout=(30, 1800),
        )
    if not response.ok:
        try:
            details = response.json().get("error", {}).get("message", response.text)
        except ValueError:
            details = response.text
        raise RuntimeError(f"Facebook rechazó el video: {details}")
    video_id = response.json().get("id")
    if not video_id:
        raise RuntimeError("Facebook respondió sin un identificador de video.")
    return str(video_id)


def run_job(job_id: str) -> None:
    with render_lock:
        with jobs_lock:
            job = jobs[job_id]
            job["status"] = "running"
            job["message"] = "Renderizando con h264_nvenc"
        work_dir = Path(job["work_dir"])
        payload = job["payload"]
        try:
            story = clean_text(job["story"])
            if not story:
                raise ValueError("La narración recibida está vacía.")
            output_path = work_dir / str(payload["output_name"])
            render_video(
                story,
                Path(job["audio_path"]),
                Path(job["timings_path"]) if job.get("timings_path") else None,
                Path(job["background_path"]) if job.get("background_path") else None,
                Path(job["font_path"]) if job.get("font_path") else None,
                [Path(path) for path in job.get("overlay_paths", [])],
                output_path,
                payload.get("style", {}),
                lambda value: update_job(
                    job_id,
                    progress=20 + min(55.0, max(0.0, value)),
                    message="Renderizando video",
                ),
            )
            if payload["destination"] == "facebook":
                update_job(job_id, message="Publicando en Facebook", progress=80)
                video_id = publish_video(output_path, str(payload["description"]))
                update_job(
                    job_id,
                    status="completed",
                    progress=100,
                    video_id=video_id,
                    filename=output_path.name,
                    work_dir=None,
                    message="Publicado en Facebook",
                )
                try:
                    shutil.rmtree(work_dir)
                except OSError as error:
                    update_job(job_id, cleanup_warning=str(error))
                    print(f"No se pudieron eliminar todos los temporales del trabajo {job_id}: {error}")
            else:
                update_job(
                    job_id,
                    status="completed",
                    progress=100,
                    filename=output_path.name,
                    message="Video listo para descargar",
                )
        except Exception as error:
            traceback.print_exc()
            shutil.rmtree(work_dir, ignore_errors=True)
            update_job(
                job_id,
                status="failed",
                progress=100,
                error=str(error),
                work_dir=None,
                message="El trabajo falló",
            )
        finally:
            gc.collect()


@app.get("/health")
def health(x_api_key: str | None = Header(None)) -> dict[str, str]:
    authorized(x_api_key)
    return {"status": "ready", "encoder": "h264_nvenc"}


@app.post("/jobs")
async def create_job(
    background_tasks: BackgroundTasks,
    payload: str = Form(...),
    audio: UploadFile = File(...),
    timings: UploadFile | None = File(None),
    background: UploadFile | None = File(None),
    font: UploadFile | None = File(None),
    overlay: UploadFile | None = File(None),
    overlays: list[UploadFile] | None = File(None),
    x_api_key: str | None = Header(None),
) -> dict[str, str]:
    authorized(x_api_key)
    try:
        data = json.loads(payload)
    except json.JSONDecodeError as error:
        raise HTTPException(status_code=400, detail="El campo payload debe contener JSON válido.") from error
    if not isinstance(data, dict) or data.get("destination") not in {"facebook", "local"}:
        raise HTTPException(status_code=400, detail="El destino debe ser facebook o local.")
    story = str(data.get("story", "")).strip()
    if not story or len(story) > 2_000_000:
        raise HTTPException(status_code=400, detail="La narración está vacía o excede el tamaño permitido.")
    style = data.get("style", {})
    if not isinstance(style, dict):
        raise HTTPException(status_code=400, detail="El estilo del video no es válido.")
    safe_name = re.sub(r"[^A-Za-z0-9_.-]", "_", str(data.get("output_name", "historia.mp4")))
    if not safe_name.lower().endswith(".mp4"):
        safe_name += ".mp4"
    data["output_name"] = safe_name
    if data["destination"] == "facebook" and not str(data.get("description", "")).strip():
        raise HTTPException(status_code=400, detail="Falta la descripción para Facebook.")

    with jobs_lock:
        has_pending = any(
            job.get("status") in {"receiving", "queued", "running"}
            or (
                job.get("status") == "completed"
                and job.get("payload", {}).get("destination") == "local"
                and job.get("work_dir")
            )
            for job in jobs.values()
        )
        if has_pending:
            raise HTTPException(
                status_code=429,
                detail="Colab procesa un video a la vez; espera a que termine la descarga actual.",
            )
        job_id = secrets.token_urlsafe(18)
        work_dir = Path(tempfile.mkdtemp(prefix=f"job_{job_id}_", dir=WORK_ROOT))
        jobs[job_id] = {
            "status": "receiving",
            "progress": 0,
            "message": "Recibiendo archivos",
            "payload": data,
            "work_dir": str(work_dir),
        }
    try:
        audio_path = work_dir / "narration.mp3"
        await save_upload(audio, audio_path, 1_500_000_000)
        job = {
            "status": "queued",
            "progress": 0,
            "message": "En cola",
            "payload": data,
            "story": story,
            "work_dir": str(work_dir),
            "audio_path": str(audio_path),
            "timings_path": None,
            "background_path": None,
            "font_path": None,
        }
        for upload, key, filename, limit, allowed_suffixes in (
            (timings, "timings_path", "timings.json", 20_000_000, {".json"}),
            (
                background,
                "background_path",
                "background",
                1_500_000_000,
                {".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi", ".jpg", ".jpeg", ".png", ".webp", ".bmp"},
            ),
            (font, "font_path", "subtitle_font", 50_000_000, {".ttf", ".otf", ".ttc"}),
        ):
            if upload is not None:
                suffix = Path(upload.filename or "").suffix.lower()
                if suffix not in allowed_suffixes:
                    raise HTTPException(
                        status_code=400,
                        detail=f"Tipo de archivo no permitido para {filename}: {suffix or '(sin extensión)'}",
                    )
                destination = work_dir / (filename if key == "timings_path" else f"{filename}{suffix}")
                await save_upload(upload, destination, limit)
                job[key] = str(destination)
        overlay_uploads = ([overlay] if overlay is not None else []) + (overlays or [])
        overlay_paths = []
        for index, upload in enumerate(overlay_uploads):
            suffix = Path(upload.filename or "").suffix.lower()
            if suffix not in {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}:
                raise HTTPException(
                    status_code=400,
                    detail=f"Tipo de archivo no permitido para overlay: {suffix or '(sin extensión)'}",
                )
            destination = work_dir / f"overlay_{index}{suffix}"
            await save_upload(upload, destination, 100_000_000)
            overlay_paths.append(str(destination))
        job["overlay_paths"] = overlay_paths
        with jobs_lock:
            jobs[job_id].update(job)
        background_tasks.add_task(run_job, job_id)
    except Exception:
        shutil.rmtree(work_dir, ignore_errors=True)
        with jobs_lock:
            jobs.pop(job_id, None)
        raise
    finally:
        await audio.close()
        if timings is not None:
            await timings.close()
        if background is not None:
            await background.close()
        if font is not None:
            await font.close()
        if overlay is not None:
            await overlay.close()
        for upload in overlays or []:
            await upload.close()
    return {"job_id": job_id, "status": "queued"}


@app.get("/jobs/{job_id}")
def job_status(job_id: str, x_api_key: str | None = Header(None)) -> dict[str, Any]:
    authorized(x_api_key)
    with jobs_lock:
        job = jobs.get(job_id)
        if not job:
            raise HTTPException(status_code=404, detail="No existe ese trabajo.")
        return {
            key: job.get(key)
            for key in ("status", "progress", "message", "video_id", "filename", "error")
        }


@app.get("/jobs/{job_id}/download")
def download_job(job_id: str, x_api_key: str | None = Header(None)) -> FileResponse:
    authorized(x_api_key)
    with jobs_lock:
        job = jobs.get(job_id)
        if not job or job.get("status") != "completed" or job["payload"]["destination"] != "local":
            raise HTTPException(status_code=404, detail="El video no está disponible para descargar.")
        file_path = Path(job["work_dir"]) / str(job["filename"])
        if not file_path.is_file():
            raise HTTPException(status_code=404, detail="El archivo renderizado ya no está disponible.")
    return FileResponse(
        file_path,
        media_type="video/mp4",
        filename=str(job["filename"]),
        background=BackgroundTask(cleanup_download, job_id),
    )


def cleanup_download(job_id: str) -> None:
    with jobs_lock:
        job = jobs.get(job_id)
        work_dir = job.get("work_dir") if job else None
        if job:
            job["work_dir"] = None
    if work_dir:
        try:
            shutil.rmtree(work_dir)
        except OSError as error:
            print(f"No se pudieron eliminar todos los temporales del trabajo {job_id}: {error}")


print(f"GPU/FFmpeg listos: {FFMPEG_PATH}")
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(60):
    if server.started:
        break
    time.sleep(0.5)
else:
    raise RuntimeError("Uvicorn no inició dentro del tiempo esperado.")
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url.replace("http://", "https://", 1)
print("\nServidor Colab listo.")
print(f"URL pública: {PUBLIC_URL}")
print(f"Clave API para la aplicación local: {API_KEY}")
print("No cierres esta sesión mientras se estén procesando videos.")